In [ ]:
import sys, os, importlib, warnings
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # CTC loss가 MPS 미지원 → CPU fallback
sys.path.insert(0, "/Users/sknoh/Documents/Workspace/aiscore")
os.chdir("/Users/sknoh/Documents/Workspace/aiscore")

# MPS fallback UserWarning 억제
warnings.filterwarnings("ignore", category=UserWarning, message=".*aten::_ctc_loss.*")

# 모듈 재로드 → train_omr.py 수정사항(print 등) 즉시 반영
import training.scripts.train_omr as _m
importlib.reload(_m)
_m.train(epochs=30)

[학습 시작] 디바이스=mps  train=1265  val=158  epochs=30
------------------------------------------------------------


In [ ]:
# 학습 결과 확인
import torch
ckpt = torch.load("training/models/omr_crnn_best.pt", map_location="cpu", weights_only=False)
print(f"Best val_loss : {ckpt['val_loss']:.4f}")
print(f"저장된 epoch  : {ckpt['epoch']}")

In [ ]:
# 추론 샘플 (소프라노 앞 10음 예측 vs 정답)
import json
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torch
from training.scripts.train_omr import OmrCRNN, NoteVocab, IMG_H, IMG_W

vocab = NoteVocab()
model = OmrCRNN(vocab.size)
ckpt  = torch.load("training/models/omr_crnn_best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

splits = json.loads(Path("training/data/splits.json").read_text())
label  = json.loads(Path(splits["val"][0]["label_path"]).read_text())
img    = Image.open(label["image_path"]).convert("RGB")

transform = T.Compose([
    T.Grayscale(3), T.Resize((IMG_H, IMG_W)), T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])
x = transform(img).unsqueeze(0)

with torch.no_grad():
    logits = model(x)

# CTC greedy decode
def ctc_decode(lgt):
    ids = lgt.argmax(dim=-1).squeeze(1).tolist()
    out, prev = [], None
    for i in ids:
        if i != vocab.blank_idx and i != prev:
            out.append(i)
        prev = i
    return vocab.decode(out)

pred_s = ctc_decode(logits["S"])
gt_s   = [n for m in label["measures"] for n in m["S"]]

print(f"곡: hymn{label['hymn_id']} system={label.get('system','-')}  "
      f"정답 {len(gt_s)}음 / 예측 {len(pred_s)}음")
print(f"{'#':>3}  {'정답pitch':>10}  {'예측pitch':>12}  일치")
for i in range(min(10, len(gt_s), len(pred_s))):
    ok = "✅" if pred_s[i]["pitch"] == gt_s[i]["pitch"] else "❌"
    print(f"{i+1:>3}  {gt_s[i]['pitch']:>10}  {pred_s[i]['pitch']:>12}  {ok}")
if not pred_s:
    print("(예측 없음 — 모델이 blank만 출력)")